#### Importing libraries

In [2]:
import pandas as pd
import numpy as np
import seaborn as sns
import os
import operator
import time
import matplotlib.pyplot as plt
from sklearn.preprocessing import LabelEncoder
import tensorflow as tf
from numpy import unique
from numpy import reshape
from keras.models import Sequential
from bayes_opt import BayesianOptimization
from sklearn.model_selection import cross_val_score
from keras.layers import Conv1D, Conv2D, Dense, Dropout, BatchNormalization, Flatten, MaxPooling1D
from keras.layers import Dense, Dropout
from keras.optimizers import Adam, SGD, RMSprop, Adadelta, Adagrad, Adamax, Nadam, Ftrl
from keras.callbacks import EarlyStopping, ModelCheckpoint
from math import floor
from sklearn.metrics import make_scorer, accuracy_score
from sklearn.model_selection import StratifiedKFold
from keras.layers import LeakyReLU
LeakyReLU = LeakyReLU(alpha=0.1)
import warnings
warnings.filterwarnings('ignore')
pd.set_option("display.max_columns", None)
from sklearn.model_selection import train_test_split

C:\Users\artoe\anaconda3\Lib\site-packages\keras\src\layers\activations\leaky_relu.py:41: UserWarning: Argument `alpha` is deprecated. Use `negative_slope` instead.
  warnings.warn(


In [3]:
pip install np_utils

Note: you may need to restart the kernel to use updated packages.


In [4]:
pip install SciKeras

Note: you may need to restart the kernel to use updated packages.


In [5]:
from scikeras.wrappers import KerasClassifier

#### Set path, import datasets

In [7]:
#set file path for data
path = r'/Users/artoe/Documents/DataAnalytics/Machine Learning With Python/Achievement 2/data'

In [8]:
#import cleaned, unscaled weather data
weather = pd.read_csv(os.path.join(path, 'weather_cleaned.csv'), index_col = False)

In [9]:
#load pleasant weather answers data
pleasant = pd.read_csv(os.path.join(path, 'Pleasant_Weather.csv'), index_col = False)

In [10]:
#set tf random seed to ensure reproducible results
tf.random.set_seed(42)

In [11]:
#drop date/date+month columns
weather.drop(columns=['DATE','MONTH'], inplace=True)

In [12]:
pleasant.drop(columns='DATE', inplace=True)

In [13]:
weather.shape

(22950, 135)

In [14]:
pleasant.shape

(22950, 15)

#### Reshape and reformat data for deep learning

In [16]:
#rename dataframes so that the weather data is X and the pleasant weather answers are y
X = weather
y = pleasant

In [17]:
#convert these to arrays
X = np.array(X)
y = np.array(y)

In [18]:
X.shape

(22950, 135)

In [19]:
y.shape

(22950, 15)

In [20]:
#the X set has 22950 rows with 90 columns. 
#the 135 columns consist of 9 observations (cloud cover, global radiation, humidity, precipitation, pressure, sunshine, mean-, min- and max-temperature) for 15 locations.
X = X.reshape(-1,15,9)

In [21]:
X.shape

(22950, 15, 9)

In [22]:
#we need to reshape y so that it can work with the Bayesian Optimizer
#first we will check the indicator version
from sklearn.utils.multiclass import type_of_target
type_of_target(y)

'multilabel-indicator'

In [23]:
#using np.argmax, we change the format to multiclass and supply the numerical value
y = np.argmax(y, axis = 1)
print(y.shape)
y

(22950,)


array([0, 0, 0, ..., 0, 0, 0], dtype=int64)

In [24]:
#checking the type again
type_of_target(y)

'multiclass'

In [25]:
#split into train and test sets
X_train, X_test, y_train, y_test = train_test_split(X,y,random_state=42)

In [26]:
print(X_train.shape, y_train.shape)
print(X_test.shape, y_test.shape)

(17212, 15, 9) (17212,)
(5738, 15, 9) (5738,)


In [27]:
#check for nulls in y
nulls = np.isnan(y_train)

In [28]:
np.unique(nulls)

array([False])

In [29]:
nulls2 = np.isnan(y_test)

In [30]:
np.unique(nulls2)

array([False])

### Optimize Parameters with Bayesian Function

#### Initial Parameters

In [33]:
timesteps = len(X_train[0])
input_dim = len(X_train[0][0])
n_classes = 15
# Make scorer accuracy
score_acc = make_scorer(accuracy_score)

In [34]:
# Create function
def bay_area(neurons, activation, kernel, optimizer, learning_rate, batch_size, epochs,
              layers1, layers2, normalization, dropout, dropout_rate): 
    optimizerL = ['SGD', 'Adam', 'RMSprop', 'Adadelta', 'Adagrad', 'Adamax', 'Nadam', 'Ftrl','SGD']
    optimizerD= {'Adam':Adam(learning_rate=learning_rate), 'SGD':SGD(learning_rate=learning_rate),
                 'RMSprop':RMSprop(learning_rate=learning_rate), 'Adadelta':Adadelta(learning_rate=learning_rate),
                 'Adagrad':Adagrad(learning_rate=learning_rate), 'Adamax':Adamax(learning_rate=learning_rate),
                 'Nadam':Nadam(learning_rate=learning_rate), 'Ftrl':Ftrl(learning_rate=learning_rate)}
    activationL = ['relu', 'sigmoid', 'softplus', 'softsign', 'tanh', 'selu',
                   'elu', 'exponential', LeakyReLU,'relu']
    
    neurons = round(neurons)
    kernel = round(kernel)
    activation = activationL[round(activation)]
    optimizer = optimizerL[round(optimizer)] #optimizerD[optimizerL[round(optimizer)]]
    batch_size = round(batch_size)
    
    epochs = round(epochs)
    layers1 = round(layers1)
    layers2 = round(layers2)
    
    def cnn_model():
        model = Sequential()
        model.add(Conv1D(neurons, kernel_size=kernel,activation=activation, input_shape=(timesteps, input_dim)))
        #model.add(Conv1D(32, kernel_size=1,activation='relu', input_shape=(timesteps, input_dim)))
        
        if normalization > 0.5:
            model.add(BatchNormalization())
        for i in range(layers1):
            model.add(Dense(neurons, activation=activation)) #(neurons, activation=activation))
        if dropout > 0.5:
            model.add(Dropout(dropout_rate, seed=123))
        for i in range(layers2):
            model.add(Dense(neurons, activation=activation))
        model.add(MaxPooling1D())
        model.add(Flatten())
        model.add(Dense(n_classes, activation='softmax')) #sigmoid softmax
        #model.compile(loss='binary_crossentropy', optimizer=optimizer, metrics=['accuracy']) #categorical_crossentropy
        model.compile(loss='sparse_categorical_crossentropy', optimizer=optimizer, metrics=['accuracy']) #categorical_crossentropy
        return model
    es = EarlyStopping(monitor='accuracy', mode='max', verbose=2, patience=20)
    nn = KerasClassifier(build_fn=cnn_model, epochs=epochs, batch_size=batch_size, verbose=2)
    kfold = StratifiedKFold(n_splits=5, shuffle=True, random_state=123)
    score = cross_val_score(nn, X_train, y_train, scoring=score_acc, cv=kfold, fit_params={'callbacks':[es]}).mean()
    return score

In [35]:
start = time.time()
params ={
    'neurons': (10, 100),
    'kernel': (1, 3),
    'activation':(0, 9), #9
    'optimizer':(0,7), #7
    'learning_rate':(0.01, 1),
    'batch_size': (200, 1000), #(10, 50), #
    'epochs':(20, 100),
    'layers1':(1,3),
    'layers2':(1,3),
    'normalization':(0,1),
    'dropout':(0,1),
    'dropout_rate':(0,0.3)
}
# Run Bayesian Optimization
nn_opt = BayesianOptimization(bay_area, params, random_state=42)
nn_opt.maximize(init_points=15, n_iter=4) #25
print('Search took %s minutes' % ((time.time() - start)/60))

|   iter    |  target   | activa... | batch_... |  dropout  | dropou... |  epochs   |  kernel   |  layers1  |  layers2  | learni... |  neurons  | normal... | optimizer |
-------------------------------------------------------------------------------------------------------------------------------------------------------------------------
Epoch 1/32
15/15 - 2s - 138ms/step - accuracy: 0.5976 - loss: 2.7180
Epoch 2/32
15/15 - 0s - 31ms/step - accuracy: 0.6440 - loss: 2.7004
Epoch 3/32
15/15 - 0s - 31ms/step - accuracy: 0.6440 - loss: 2.6971
Epoch 4/32
15/15 - 0s - 31ms/step - accuracy: 0.6440 - loss: 2.6942
Epoch 5/32
15/15 - 0s - 31ms/step - accuracy: 0.6440 - loss: 2.6917
Epoch 6/32
15/15 - 0s - 31ms/step - accuracy: 0.6440 - loss: 2.6894
Epoch 7/32
15/15 - 0s - 31ms/step - accuracy: 0.6440 - loss: 2.6873
Epoch 8/32
15/15 - 0s - 30ms/step - accuracy: 0.6440 - loss: 2.6853
Epoch 9/32
15/15 - 0s - 31ms/step - accuracy: 0.6440 - loss: 2.6834
Epoch 10/32
15/15 - 1s - 35ms/step - accuracy: 

KeyboardInterrupt: 

In [36]:
optimum = nn_opt.max['params']
learning_rate = optimum['learning_rate']
activationL = ['relu', 'sigmoid', 'softplus', 'softsign', 'tanh', 'selu',
               'elu', 'exponential', LeakyReLU,'relu']
optimum['activation'] = activationL[round(optimum['activation'])]
optimum['batch_size'] = round(optimum['batch_size'])
optimum['epochs'] = round(optimum['epochs'])
optimum['layers1'] = round(optimum['layers1'])
optimum['layers2'] = round(optimum['layers2'])
optimum['neurons'] = round(optimum['neurons'])
optimizerL = ['Adam', 'SGD', 'RMSprop', 'Adadelta', 'Adagrad', 'Adamax', 'Nadam', 'Ftrl','Adam']
optimizerD= {'Adam':Adam(learning_rate=learning_rate), 'SGD':SGD(learning_rate=learning_rate),
             'RMSprop':RMSprop(learning_rate=learning_rate), 'Adadelta':Adadelta(learning_rate=learning_rate),
             'Adagrad':Adagrad(learning_rate=learning_rate), 'Adamax':Adamax(learning_rate=learning_rate),
             'Nadam':Nadam(learning_rate=learning_rate), 'Ftrl':Ftrl(learning_rate=learning_rate)}
optimum['optimizer'] = optimizerD[optimizerL[round(optimum['optimizer'])]]
optimum

{'activation': 'tanh',
 'batch_size': 828,
 'dropout': 0.19967378215835974,
 'dropout_rate': 0.15427033152408348,
 'epochs': 67,
 'kernel': 1.0929008254399954,
 'layers1': 2,
 'layers2': 1,
 'learning_rate': 0.07440107705542671,
 'neurons': 95,
 'normalization': 0.9656320330745594,
 'optimizer': <keras.src.optimizers.nadam.Nadam at 0x2e42866df40>}

### Run CNN Model with Optimized Hyperparameters

In [39]:
#change y_train and y_test back to one-hot encoded format
from keras.utils import to_categorical

In [41]:
y_train = to_categorical(y_train)

In [43]:
y_test = to_categorical(y_test)

In [45]:
y_train.shape

(17212, 15)

In [47]:
y_test.shape

(5738, 15)

In [49]:
#check type once more 
type_of_target(y_train)

'multilabel-indicator'

In [51]:
type_of_target(y_test)

'multilabel-indicator'

In [53]:
#both y sets are back in one-hot encoded format and can be used with the CNN model

In [116]:
#create model
epochs = 67
batch_size = 828
n_hidden = 32
timesteps = len(X_train[0])
input_dim = len(X_train[0][0])
n_classes = 15
layers1 = 2
layers2 = 1
activation = 'tanh'
kernel = 1
neurons = 95
normalization = 0.9656320330745594
dropout = 0.19967378215835974
dropout_rate = 0.15427033152408348
optimizer = 'Nadam' 
learning_rate: 0.07440107705542671

model = Sequential()
model.add(Conv1D(neurons, kernel_size=kernel, activation=activation, input_shape=(timesteps, input_dim)))
if normalization > 0.5:
    model.add(BatchNormalization())
for i in range(layers1):
    model.add(Dense(neurons, activation=activation))
if dropout > 0.5:
    model.add(Dropout(dropout_rate, seed=123))
for i in range(layers2):
    model.add(Dense(neurons, activation=activation))
model.add(MaxPooling1D())
model.add(Flatten())
model.add(Dense(n_classes, activation='softmax')) #softmax sigmoid
model.compile(loss='sparse_categorical_crossentropy', optimizer=optimizer, metrics=['accuracy'])

In [118]:
model.summary()

Model: "sequential_88"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                         ┃ Output Shape                ┃         Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━┩
│ conv1d_88 (Conv1D)                   │ (None, 15, 95)              │             950 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ batch_normalization_39               │ (None, 15, 95)              │             380 │
│ (BatchNormalization)                 │                             │                 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense_440 (Dense)                    │ (None, 15, 95)              │           9,120 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense_441 (Dense)                    │ (None, 15, 95)              │           9,120 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense_442 (Dense)                    │ (None, 15, 95)              │           9,120 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ max_pooling1d_88 (MaxPooling1D)      │ (None, 7, 95)               │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ flatten_88 (Flatten)                 │ (None, 665)                 │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense_443 (Dense)                    │ (None, 15)                  │           9,990 │
└──────────────────────────────────────┴─────────────────────────────┴─────────────────┘

 Total params: 38,680 (151.09 KB)

 Trainable params: 38,490 (150.35 KB)

 Non-trainable params: 190 (760.00 B)

In [120]:
model.compile(loss='categorical_crossentropy', optimizer='adam', metrics=['accuracy'])

In [122]:
model.fit(X_train, y_train, batch_size=batch_size, epochs=epochs, verbose=2)

Epoch 1/67
21/21 - 2s - 102ms/step - accuracy: 0.6093 - loss: 1.2326
Epoch 2/67
21/21 - 1s - 34ms/step - accuracy: 0.7018 - loss: 0.8721
Epoch 3/67
21/21 - 1s - 35ms/step - accuracy: 0.7361 - loss: 0.7917
Epoch 4/67
21/21 - 1s - 35ms/step - accuracy: 0.7538 - loss: 0.7397
Epoch 5/67
21/21 - 1s - 36ms/step - accuracy: 0.7669 - loss: 0.6978
Epoch 6/67
21/21 - 1s - 36ms/step - accuracy: 0.7766 - loss: 0.6625
Epoch 7/67
21/21 - 1s - 38ms/step - accuracy: 0.7863 - loss: 0.6318
Epoch 8/67
21/21 - 1s - 35ms/step - accuracy: 0.7933 - loss: 0.6029
Epoch 9/67
21/21 - 1s - 41ms/step - accuracy: 0.8019 - loss: 0.5762
Epoch 10/67
21/21 - 1s - 38ms/step - accuracy: 0.8106 - loss: 0.5517
Epoch 11/67
21/21 - 1s - 37ms/step - accuracy: 0.8168 - loss: 0.5290
Epoch 12/67
21/21 - 1s - 35ms/step - accuracy: 0.8227 - loss: 0.5087
Epoch 13/67
21/21 - 1s - 34ms/step - accuracy: 0.8270 - loss: 0.4910
Epoch 14/67
21/21 - 1s - 34ms/step - accuracy: 0.8328 - loss: 0.4758
Epoch 15/67
21/21 - 1s - 36ms/step - accur

##### Accuracy of 93.5%, loss of 0.191

#### Show accuracy of model with confusion matrix

In [126]:
#set location names
locations = {
    0: 'BASEL',
    1: 'BELGRADE',
    2: 'BUDAPEST',
    3: 'DEBILT',
    4: 'DUSSELDORF',
    5: 'HEATHROW',
    6: 'KASSEL',
    7: 'LJUBLJANA',
    8: 'MAASTRICHT',
    9: 'MADRID',
    10: 'MUNCHENB',
    11: 'OSLO',
    12: 'SONNBLICK',
    13: 'STOCKHOLM',
    14: 'VALENTIA',
}

In [128]:
def confusion_matrix(Y_true, Y_pred):
    Y_true = pd.Series([locations[y] for y in np.argmax(Y_true, axis=1)])
    Y_pred = pd.Series([locations[y] for y in np.argmax(Y_pred, axis=1)])

    return pd.crosstab(Y_true, Y_pred, rownames=['True'], colnames=['Pred'])

In [130]:
#evaluate
print(confusion_matrix(y_test, model.predict(X_test)))

180/180 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step
Pred        BASEL  BELGRADE  BUDAPEST  DEBILT  DUSSELDORF  HEATHROW  KASSEL  \
True                                                                          
BASEL        3561        66         6      10           7         7       0   
BELGRADE      120       944         7       4           1         2       0   
BUDAPEST       19        23       146      12           1         5       0   
DEBILT          9         5         7      56           2         1       0   
DUSSELDORF      4         2         1       2           5        13       0   
HEATHROW        8         2         1       1           5        55       0   
KASSEL          5         2         1       0           3         0       0   
LJUBLJANA      12         2         4       0           0         2       1   
MAASTRICHT      2         0         0       0           0         1       0   
MADRID         38        11         8       0           0         5       2   
MUNCHENB   